# DNC's insults vs Trump's own tweets

The DNC's account calls Trump "dumbass," "stupid," "gross," "ugly." How many of those are the account's own invention, and how many is it borrowing from Trump's own decade of insults?

The insult set is a curated lexicon (`curated_insults.csv`) I pulled by hand from the vetted "according to Dems" walls — every insult the account aimed at Trump and Vance, checked post by post against the video. Then I test each one two ways: does the word show up anywhere in Trump's ~46,000 archived tweets, and — separately — did Trump ever wield it as a personal insult himself, rather than a stray or quoted use? That second call is the harder one that a one-off LLM pass made - verdicts here: `trump_usage_llm.jsonl`. Both results live in `insult_lineage_audit_table.csv`, which this notebook reads.

## Setup

In [1]:
import pandas as pd
pd.set_option("display.max_rows", None)       # show every row — no truncated lists
pd.set_option("display.max_colwidth", None)   # show full cell text
from pathlib import Path

ROOT = Path.cwd()
for cand in (ROOT, *ROOT.parents):
    if (cand / "data" / "analysis" / "insult_lineage_audit_table.csv").exists():
        ROOT = cand; break
AN = ROOT / "data" / "analysis"

audit   = pd.read_csv(AN / "insult_lineage_audit_table.csv")
curated = pd.read_csv(AN / "curated_insults.csv")
usage   = pd.read_json(AN / "trump_usage_llm.jsonl", lines=True)
dunks   = pd.read_csv(AN / "dunk_lines.csv")

print(f"{len(curated)} curated insults; {len(audit)} rows in the audit table")
audit.head(4)

100 curated insults; 96 rows in the audit table


,term,target,register,in_trump_lexically,trump_weaponized,trump_verdict,trump_evidence,own_voice_uses,sample_dem_use,sample_post_id,note
0,ugly,trump,body,True,True,insult_at_person,@ noma2300 You must be kidding - I kicked both of their ugly assses!,8,Ugly ass truck,7.480658e+18,NaN
1,fat,trump,body,True,True,insult_at_person,"Why would Kim Jong-un insult me by calling me 'old,' when I would NEVER call him 'short and fat?' Oh well, I try so…",11,"You've called women you don't like, ""fat pigs,""",7.418643e+18,NaN
2,fatty,trump,body,False,False,never_tweeted,NaN,1,fatty alert,7.636513e+18,NaN
3,tiny,trump,body,True,True,insult_at_person,Mini Mike is a short ball (very) hitter. Tiny club head speed. KEEP AMERICA GREAT! https://t.co/5DUj16jtZf,1,why does Trump look so tiny 😂,7.526391e+18,NaN


## §1 — The curated lexicon, by register

The lexicon is hand-pulled from the vetted walls, not from a keyword scanner — an earlier ~25-word scan missed most of it. I group it by register (body / mind / morals, plus a Vance-only subservience set), because that grouping is what the rest of the analysis turns on.

In [2]:

curated.groupby("register")["term"].agg(n="size", terms=lambda s: ", ".join(sorted(s)))


,n,terms
register,,
body,22,"bad hair, bad posture, creature, crusty, decaying, decaying sack of flesh, decrepit, fat, fatty, gross, mouth breather, nasty, repulsive, scaly, snorlax, sweaty, tan, tiny, too many chins, ugly, wig, wrinkled"
mind,31,"80-year-old man, can't read, chopped, confused, cooked, crashing out, deteriorating, dumb, grandpa, insane, insecure, low energy, mad, melts down, mess, mogged, old, old as fart, oldest, rambling, sad sack, senile, sleepy, snoozing, stupid, unc, unhinged, unstable, unwell, useless, weird"
morals,33,"chicken, chud, convicted felon, diabolical, disgusting, epstein's bff, evil, fat chud, fat little chud, flop, haunted, horny, jealous, jobless, liar, loony bin, looter, loser, nightmare, pigging out, predator, rat, resident evil, sleep-paralysis demon, snowflake, spooky, tacky, tacky gold, traitor, true chud, vile, wannabe dictator, worst"
subservience,14,"bootlicker, concerning, creep, disgusting, dumb, irrelevant, jd-chan, mad, negative aura, owned, poor, sellout, ugly bootlicker, weird"


## The walls these come from

The lexicon above is distilled from the walls I vetted by hand: every insult @democrats aimed at Trump and Vance, pulled from captions, on-screen text and transcripts, checked post by post. Here they are in full — parsed straight from `walls_print.md`.

In [3]:
import re

text = (ROOT / "data" / "walls_print.md").read_text()
sections = {"Trump, according to Dems": "trump", "Vance, according to Dems": "vance"}

rows = []
for section in text.split("## ")[1:]:
    head = section.split("\n", 1)[0].strip()
    if head not in sections:
        continue
    for line in section.splitlines():
        m = re.match(r"\s*(20\d\d)(?:\s*\(through July\))?\s+(.*)", line)
        if not m:
            continue
        year, body = m.group(1), m.group(2)
        for snippet in re.split(r"\s{2,}", body):     # wall lines are separated by 2+ spaces
            snippet = snippet.strip(" *")
            if len(snippet) > 1 and snippet != "—":
                rows.append({"target": sections[head], "year": year, "line": snippet})

walls = pd.DataFrame(rows)
print("wall lines:", walls.groupby("target").size().to_dict())
walls

wall lines: {'trump': 264, 'vance': 26}


,target,year,line
0,trump,2023,60 seconds of Donald Trump blabbing... 🤴🦞💨🐋🤡
1,trump,2023,we're glad you lost
2,trump,2023,Flop.
3,trump,2023,THUNDA
4,trump,2023,Is Donald Trump... okay?🤡
5,trump,2024,A deeply confused Trump confuses Nancy Pelosi and Nikki Haley multiple times
6,trump,2024,oh brother this guy stinks
7,trump,2024,Them being weird
8,trump,2024,"Republicans Are Weird... These are weird people on the other side. Listen to them speak. These are weird ideas. [Trump moans] These are weird, weird people. Don't get sugarcoating."
9,trump,2024,every presidential candidate has their strange wannabe


## How the lexicon maps to the walls

Every term in the lexicon traces back to real wall lines. This counts, for each term, how many wall lines it appears in — the distilled list and its vetted source side by side. A few phrase-form terms show 0 because they read differently in the wall than as a single keyword.

In [4]:
def wall_lines_for(term, target):
    w = walls[walls["target"] == target]
    hit = w["line"].str.contains(rf"\b{re.escape(term)}\b", case=False, regex=True)
    return w.loc[hit, "line"].tolist()

curated["wall_lines"] = [len(wall_lines_for(t, tg))
                         for t, tg in zip(curated["term"], curated["target"])]

# to read the actual lines behind any term:  wall_lines_for("ugly", "trump")
curated[["term", "target", "register", "wall_lines"]].sort_values("wall_lines", ascending=False)

,term,target,register,wall_lines
53,evil,trump,morals,9
24,unhinged,trump,mind,7
50,chopped,trump,mind,6
1,fat,trump,body,5
40,old,trump,mind,5
52,mogged,trump,mind,4
31,rambling,trump,mind,4
28,crashing out,trump,mind,4
0,ugly,trump,body,4
5,decaying,trump,body,3


## §2 — Where each insult was used

`own_voice_uses` counts only the account's own caption or on-screen text — a song lyric or a quoted
clip doesn't count. The most-used words are the account's signatures.

In [5]:

(audit[audit["own_voice_uses"] > 0]
   .sort_values("own_voice_uses", ascending=False)
   [["term", "target", "register", "own_voice_uses"]]
   .head(15).reset_index(drop=True))


,term,target,register,own_voice_uses
0,chud,trump,morals,12
1,fat,trump,body,11
2,ugly,trump,body,8
3,fat chud,trump,morals,8
4,worst,trump,morals,8
5,cooked,trump,mind,7
6,weird,trump,mind,7
7,mogged,trump,mind,6
8,evil,trump,morals,6
9,convicted felon,trump,morals,5


## §3 — Which of these words also show up in Trump's tweets

`in_trump_lexically` just means the word appears at least once in Trump's archived tweets. It's presence only — it doesn't yet mean he used it as an insult (§3b sorts that out). Obvious false friends, like the surname "Tan" for "tan," were pulled out by hand when the table was built.

In [6]:

print("Curated insults appearing lexically in Trump's tweets:",
      f"{audit['in_trump_lexically'].sum()} of {len(audit)}")

# the split by register
pd.crosstab(audit["register"], audit["in_trump_lexically"], margins=True)


Curated insults appearing lexically in Trump's tweets: 54 of 96


in_trump_lexically,False,True,All
register,,,
body,11,11,22
mind,9,22,31
morals,18,15,33
subservience,4,6,10
All,42,54,96


## §3b — But did Trump actually weaponize the word?

Lexical presence over-counts. "Tiny" shows up in his tweets, but did he aim it at a person? A one-off LLM pass read his own uses and judged each shared word; the verdicts are cached. `trump_weaponized` is the final call — my overrides baked in — that Trump used the word as a personal insult, backed by a specific tweet.

In [7]:

print("Lexically present :", audit["in_trump_lexically"].sum())
print("Actually weaponized:", audit["trump_weaponized"].sum())

# the corrected split, by register
(audit[audit["trump_weaponized"]]
   .groupby("register")["term"].agg(n="size", terms=lambda s: ", ".join(sorted(s))))


Lexically present : 54
Actually weaponized: 37


,n,terms
register,,
body,7,"creature, decaying, fat, gross, nasty, tiny, ugly"
mind,15,"can't read, confused, dumb, insane, insecure, low energy, mad, rambling, sad sack, sleepy, stupid, unhinged, unstable, useless, weird"
morals,10,"disgusting, evil, flop, jealous, liar, loser, rat, traitor, vile, worst"
subservience,5,"creep, irrelevant, owned, poor, sellout"


In [8]:

# a few weaponized words with Trump's own supporting tweet
(usage[usage["used_as_insult"]]
   .merge(audit[["term"]], left_on="word", right_on="term")
   [["word", "evidence_tweet"]].head(8))


,word,evidence_tweet
0,ugly,@ noma2300 You must be kidding - I kicked both of their ugly assses!
1,fat,"Why would Kim Jong-un insult me by calling me 'old,' when I would NEVER call him 'short and fat?' Oh well, I try so…"
2,tiny,Mini Mike is a short ball (very) hitter. Tiny club head speed. KEEP AMERICA GREAT! https://t.co/5DUj16jtZf
3,decaying,Crazy Nancy Pelosi should spend more time in her her decaying city and less time on the Impeachment Hoax!
4,gross,"The @ HuffingtonPost is a total joke & laughing stock of journalism, as is gross Arianna Huffington. They don‚Äôt report the facts!"
5,nasty,"All polls have me winning debate big- Drudge, TIME, etc. Dopey Charles Krauthammer still nasty. He has zero cred- totally dishonest!"
6,creature,"Two months in jail for a Swamp Creature, yet 9 years recommended for Roger Stone (who was not even working for the Trump Campaign). Gee, that sounds very fair! Rogue prosecutors maybe? The Swamp! @foxandfriends @TuckerCa"
7,confused,Hillary Clinton answered email questions differently last night than she has in the past. She is totally confused. Unfit to serve as # POTUS.


## §3d — The wall: words the DNC used on Trump that Trump used on people himself

This is the "what went around came around" set — insults aimed at Trump that Trump himself weaponized. Two words pass the mechanical test but come out on review, with the reason next to each:

- **insane** — the account never cleanly calls *Trump* insane; its uses are crowd praise ("insane lineup") or a Vance line, so the Democratic side of the pairing fails.
- **evil** — Trump's cleanest "evil" aims at the media / "Fake News," not a person, and the tweet turns positive; it isn't a clean personal insult.

That leaves the 30 the story stands on.

In [9]:

# the two reviewed drops, with the reason kept beside each
DROP = {
    "insane": "account's uses are crowd praise / a Vance line, not a clean shot at Trump",
    "evil":   "Trump's 'evil' aims at the media, not a person; not a clean personal insult",
}
weaponized = audit[(audit["target"] == "trump") & (audit["trump_weaponized"])]
wall = weaponized[~weaponized["term"].isin(DROP)]

print(f"{len(weaponized)} words passed the mechanical test;",
      f"{len(DROP)} dropped on review; {len(wall)} in the wall.\n")
print("dropped on review:")
for term, reason in DROP.items():
    print(f"   x {term}: {reason}")

wall[["term", "register", "own_voice_uses", "sample_dem_use"]].reset_index(drop=True)


32 words passed the mechanical test; 2 dropped on review; 30 in the wall.

dropped on review:
   x insane: account's uses are crowd praise / a Vance line, not a clean shot at Trump
   x evil: Trump's 'evil' aims at the media, not a person; not a clean personal insult


,term,register,own_voice_uses,sample_dem_use
0,ugly,body,8,Ugly ass truck
1,fat,body,11,"You've called women you don't like, ""fat pigs,"""
2,tiny,body,1,why does Trump look so tiny 😂
3,decaying,body,2,"weird, scaly neck rash and rapidly decaying hand"
4,gross,body,1,gross
5,nasty,body,2,how trump feels going outside with his nasty rash and hand bruise
6,creature,body,1,WHAT IS THAT CREATURE?
7,confused,mind,2,*Confused*
8,unhinged,mind,4,UNHINGED
9,unstable,mind,3,UNSTABLE


## §4 — The full per-insult audit table

Every curated insult with all its verdicts in one place — lexical presence, whether Trump weaponized it, the own-voice count, a Trump evidence tweet, and any note. This is the spine; change `curated_insults.csv` and rebuild to update everything downstream.

In [10]:

audit[["term", "target", "register", "in_trump_lexically", "trump_weaponized",
       "own_voice_uses", "note"]]


,term,target,register,in_trump_lexically,trump_weaponized,own_voice_uses,note
0,ugly,trump,body,True,True,8,NaN
1,fat,trump,body,True,True,11,NaN
2,fatty,trump,body,False,False,1,NaN
3,tiny,trump,body,True,True,1,NaN
4,scaly,trump,body,False,False,1,NaN
5,decaying,trump,body,True,True,2,NaN
6,decrepit,trump,body,True,False,1,NaN
7,wrinkled,trump,body,False,False,0,NaN
8,too many chins,trump,body,False,False,0,NaN
9,bad posture,trump,body,False,False,0,NaN


## §5 — Do these words show up on the RNC feed too?

If the RNC uses the same words in its own voice, the vocabulary isn't uniquely Democratic. I search only the RNC's own-voice lines (caption or on-screen text, not songs or quoted clips). A raw word-match over-counts — the RNC quoting a Democrat, or an idiom like "worst" — so treat these as the screen; the genuine reuses are fewer, confirmed by hand.

In [11]:

OWN = ["caption", "overlay", "meme_text"]
rnc = dunks[(dunks["account"] == "republicans") & (dunks["source"].isin(OWN))]
rnc_text = rnc["dunk_line"].fillna("").str.lower()

hits = []
for term in curated["term"].unique():
    m = rnc_text.str.contains(rf"\b{term}\b", regex=True)
    if m.any():
        hits.append({"term": term, "rnc_own_voice_hits": int(m.sum()),
                     "example": rnc.loc[m, "dunk_line"].iloc[0]})
pd.DataFrame(hits).sort_values("rnc_own_voice_hits", ascending=False).reset_index(drop=True)


,term,rnc_own_voice_hits,example
0,stupid,4,This stupid freaking red hat!
1,fat,1,HIS STUPIDITY AND FAT ASS WHEEZING
2,confused,1,CONFUSED
3,mad,1,Stay mad
4,dumb,1,DUMB MOTHERF***** DIDN'T DESERVE TO LIVE.
5,mogged,1,mogged
6,worst,1,Trump derangement syndrome at its worst
